# Step 2c — Regression Analysis: Structural Relationships in Brazil's State-Level Trade

**Objective:** Test whether the structural indicators identified in Step 2b — product concentration (HHI), export frequency and shipment value density (USD/kg) — are systematically related across Brazilian states and stable over time.

**Relationship to Step 2b:** Step 2b introduces both regressions and presents the baseline results in the context of the broader state competitiveness analysis. This notebook extends that work — testing robustness across multiple specifications, quantifying the impact of Rio de Janeiro as a structural outlier, documenting the COVID-19 structural break and the post-2015 weakening trend, and synthesising both regressions into a unified state classification framework. The baseline results in Step 2b remain the entry point; the findings here are the deeper investigation.

**Prerequisites:** This notebook re-queries all required data independently from the database and does not depend on Step 2b dataframes being present in memory. Step 2b should be reviewed first as it provides the analytical context for the indicators used here.

---

This notebook covers:
1. Regression 1 — Export Frequency vs HHI (2c.1)
2. Regression 2 — Shipment Size vs Export Value Density (2c.2)
3. Synthesis — Comparing both regressions and their implications for state classification (2c.3)
4. Key Findings (2c.4)

Both regressions are run across five specifications to test robustness:

| Specification | Description |
|---|---|
| Baseline | All years, all geographic states |
| v2 | All years, non-geographic entries excluded at SQL level — primary reported result |
| v3 | Non-geographic excluded, COVID years (2020–2023) excluded |
| v4 | Non-geographic excluded, Rio de Janeiro excluded |
| v5 | Non-geographic excluded, Rio de Janeiro excluded, COVID years excluded — preferred structural estimate |

## Setup

This section establishes the analytical framework by defining temporal exclusions (COVID period), applying geographic filters, and standardizing regional classifications.  
It also constructs a consistent state-level dataset and visualization scheme to ensure comparability across all subsequent analyses.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

load_dotenv(dotenv_path='../.env', override=True)

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST', 'localhost')
DB_PORT     = os.getenv('DB_PORT', '5432')
DB_NAME     = os.getenv('DB_NAME')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

MAX_YEAR     = 2025
COVID_EXCLUDE = [2020, 2021, 2022, 2023]

## Region name translation applied throughout
region_name_map = {
    'REGIAO NORDESTE'    : 'Northeast',
    'REGIAO NORTE'       : 'North',
    'REGIAO SUDESTE'     : 'Southeast',
    'REGIAO CENTRO OESTE': 'Center-West',
    'REGIAO SUL'         : 'South',
}

## Non-geographic region filter applied at SQL level in all queries
GEO_FILTER = """
    AND u.nome_regiao NOT IN (
        'REGIAO NAO DECLARADA',
        'CONSUMO DE BORDO',
        'MERCADORIA NACIONALIZADA',
        'REEXPORTACAO'
    )
"""

## Load state reference table
df_state = pd.read_sql(
    f"""
    SELECT u.sigla AS uf,
           u.nome_estado AS state,
           u.nome_regiao AS region,
           COALESCE(e.exports_usd, 0) / 1e9 AS exports_usd_bn
    FROM uf u
    LEFT JOIN (
        SELECT \"SG_UF_NCM\", SUM(\"VL_FOB\") AS exports_usd
        FROM exp WHERE \"CO_ANO\" = {MAX_YEAR}
        GROUP BY \"SG_UF_NCM\"
    ) e ON u.sigla = e.\"SG_UF_NCM\"
    WHERE u.nome_regiao NOT IN (
        'REGIAO NAO DECLARADA','CONSUMO DE BORDO',
        'MERCADORIA NACIONALIZADA','REEXPORTACAO'
    )
    ORDER BY exports_usd_bn DESC
    """,
    engine
)
df_state = df_state[~df_state['state'].isin(['Nao Declarada', 'Exterior'])]
df_state['region'] = df_state['region'].map(region_name_map).fillna(df_state['region'])

## Standard region colour palette used across all charts
REGION_COLORS = {
    'Northeast'  : 'steelblue',
    'North'      : 'tomato',
    'Southeast'  : 'seagreen',
    'Center-West': 'darkorange',
    'South'      : 'mediumpurple',
}

print(f'Setup complete. MAX_YEAR={MAX_YEAR}. States loaded: {len(df_state)}')

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "None"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 2c.1 -- Regression 1: Export Frequency vs HHI

Testing whether a state's product concentration (HHI) predicts its export transaction
frequency -- the hypothesis being that states concentrated in a few bulk commodities
transact less frequently than diversified industrial states.

The relationship is examined through a pooled panel regression covering 1997-2025
with year-by-year results to test whether the relationship is stable over time.
A single-year regression is presented for context but is insufficient on its own
given the small cross-section of 27 states.

Non-geographic entries (CONSUMO DE BORDO, MERCADORIA NACIONALIZADA, REEXPORTACAO)
are excluded from the preferred specification at the SQL level. COVID-19 (2020-2023)
represents a structural break. Rio de Janeiro is flagged as a structural outlier driven
by oil port processing volumes rather than product diversification.

*Note: shipment count is derived from row count in the `exp` table rather than an
official transaction register -- liquid bulk commodities such as oil may generate
records at a different granularity than containerised or solid bulk goods, which
partly explains Rio de Janeiro's anomalously high frequency.*

### 2c.1.1 -- Data Queries (shared across all Regression 1 specifications)

In [ ]:
## Frequency query -- non-geographic excluded at SQL level
query_freq = f"""
    SELECT e."CO_ANO" AS year,
           e."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           COUNT(*) AS shipment_count,
           SUM(e."VL_FOB") AS exports_usd,
           SUM(e."KG_LIQUIDO") AS total_kg
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" <= {MAX_YEAR}
    AND u.nome_regiao NOT IN (
        'REGIAO NAO DECLARADA','CONSUMO DE BORDO',
        'MERCADORIA NACIONALIZADA','REEXPORTACAO'
    )
    GROUP BY e."CO_ANO", e."SG_UF_NCM", u.nome_estado
    ORDER BY year, shipment_count DESC
"""

## HHI query -- non-geographic excluded at SQL level
query_hhi = f"""
    SELECT year, uf,
           SUM(sector_exports * sector_exports) /
           (SUM(sector_exports) * SUM(sector_exports)) AS hhi
    FROM (
        SELECT e."CO_ANO" AS year,
               e."SG_UF_NCM" AS uf,
               s.codigo_sh2,
               SUM(e."VL_FOB") AS sector_exports
        FROM exp e
        JOIN uf u ON e."SG_UF_NCM" = u.sigla
        JOIN ncm n ON e."CO_NCM" = n.codigo_ncm
        JOIN ncm_sh s ON n.codigo_sh6 = s.codigo_sh6
        WHERE e."CO_ANO" <= {MAX_YEAR}
        AND u.nome_regiao NOT IN (
            'REGIAO NAO DECLARADA','CONSUMO DE BORDO',
            'MERCADORIA NACIONALIZADA','REEXPORTACAO'
        )
        GROUP BY e."CO_ANO", e."SG_UF_NCM", s.codigo_sh2
    ) t
    GROUP BY year, uf
    ORDER BY year, uf
"""

df_freq = pd.read_sql(query_freq, engine)
df_hhi  = pd.read_sql(query_hhi,  engine)
df_freq['exports_usd_bn'] = df_freq['exports_usd'] / 1e9

## Base panel -- used as the starting point for all v2-v5 specifications
df_panel_base = df_freq.merge(df_hhi, on=['year', 'uf'], how='inner')
df_panel_base = df_panel_base.merge(df_state[['uf', 'region']], on='uf', how='left')
df_panel_base['region'] = df_panel_base['region'].map(region_name_map).fillna(df_panel_base['region'])
df_panel_base['log_freq'] = np.log10(df_panel_base['shipment_count'])

print(f'Base panel loaded: {len(df_panel_base)} state-year observations')

### 2c.1.2 -- Helper: Run Pooled + Year-by-Year Regression

In [ ]:
def run_reg1(df, label):
    slope, intercept, r, p, se = stats.linregress(df['log_freq'], df['hhi'])
    r2 = r ** 2
    yearly = []
    for year, group in df.groupby('year'):
        if len(group) < 5: continue
        s, i, rv, pv, sev = stats.linregress(group['log_freq'], group['hhi'])
        yearly.append({'year': year, 'slope': round(s,4), 'r_squared': round(rv**2,4),
                       'p_value': round(pv,4), 'n': len(group)})
    df_y = pd.DataFrame(yearly)
    sig   = df_y[df_y['p_value'] < 0.05]
    print(f'  {label}')
    print(f'    N={len(df)}  Slope={slope:.4f}  R2={r2:.4f}  p={p:.6f}')
    print(f'    Sig years: {len(sig)}/{len(df_y)}  '
          f'Avg slope: {df_y["slope"].mean():.4f}  Avg R2: {df_y["r_squared"].mean():.4f}')
    return slope, r2, p, se, df_y

print('Helper function defined.')

### 2c.1.3 -- All Specifications: Regression 1

In [ ]:
## Build all five specifications from the base panel
v2 = df_panel_base.copy()                                                   ## all years, geo only
v3 = df_panel_base[~df_panel_base['year'].isin(COVID_EXCLUDE)].copy()       ## excl. COVID
v4 = df_panel_base[df_panel_base['uf'] != 'RJ'].copy()                      ## excl. RJ
v5 = df_panel_base[(df_panel_base['uf'] != 'RJ') &                          ## excl. RJ + COVID
                   (~df_panel_base['year'].isin(COVID_EXCLUDE))].copy()

print('Regression 1 -- all specifications')
print('-' * 70)
slope_v2, r2_v2, p_v2, se_v2, df_yearly_v2 = run_reg1(v2, 'v2 -- all years, geo regions only')
slope_v3, r2_v3, p_v3, se_v3, df_yearly_v3 = run_reg1(v3, 'v3 -- excl. COVID')
slope_v4, r2_v4, p_v4, se_v4, df_yearly_v4 = run_reg1(v4, 'v4 -- excl. RJ')
slope_v5, r2_v5, p_v5, se_v5, df_yearly_v5 = run_reg1(v5, 'v5 -- excl. RJ + COVID (preferred)')

### 2c.1.4 -- Pooled Regression Chart (v2 primary + v5 preferred structural)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

for ax, panel, slope, r2, label in [
    (ax1, v2, slope_v2, r2_v2, f'v2 -- All Years (primary)'),
    (ax2, v5, slope_v5, r2_v5, f'v5 -- Excl. RJ + COVID (structural)'),
]:
    x = np.linspace(panel['log_freq'].min(), panel['log_freq'].max(), 100)
    y = slope * x + (panel['hhi'].mean() - slope * panel['log_freq'].mean())
    for region, group in panel.groupby('region'):
        ax.scatter(group['log_freq'], group['hhi'],
                   color=REGION_COLORS.get(region, 'gray'), s=20, alpha=0.20, zorder=2)
    df_lat = panel[panel['year'] == MAX_YEAR]
    for region, group in df_lat.groupby('region'):
        ax.scatter(group['log_freq'], group['hhi'],
                   color=REGION_COLORS.get(region, 'gray'), s=60, alpha=0.95, zorder=4,
                   edgecolors='black', linewidths=0.5, label=region)
        for _, row in group.iterrows():
            ax.annotate(row['uf'], (row['log_freq'], row['hhi']), fontsize=7, ha='left', va='bottom')
    ax.plot(x, slope * x + (panel['hhi'].mean() - slope * panel['log_freq'].mean()),
            color='black', linewidth=1.8, linestyle='--',
            label=f'R2={r2:.3f}')
    ax.set_title(f'Regression 1 ({label})\nN={len(panel)}  Slope={slope:.4f}  R2={r2:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Log10 of Shipment Count')
    ax.set_ylabel('HHI Product Concentration')
    ax.legend(fontsize=7, title='Region')

plt.tight_layout()
plt.savefig('output_2c1_reg1_comparison.png', dpi=150)
plt.show()

In [ ]:
############Modify to include amazonas

### 2c.1.5 -- Year-by-Year Results (v2 and v5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=True)

for col, (df_y, slope_pool, r2_pool, label) in enumerate([
    (df_yearly_v2, slope_v2, r2_v2, 'v2 -- All Years'),
    (df_yearly_v5, slope_v5, r2_v5, 'v5 -- Excl. RJ + COVID'),
]):
    colors = ['steelblue' if p < 0.05 else 'lightgray' for p in df_y['p_value']]
    axes[0, col].bar(df_y['year'], df_y['slope'], color=colors, alpha=0.85)
    axes[0, col].axhline(y=0, color='black', linewidth=0.8)
    axes[0, col].axhline(y=slope_pool, color='tomato', linewidth=1.5, linestyle='--',
                         label=f'Pooled slope ({slope_pool:.4f})')
    axes[0, col].set_title(f'Year-by-Year Slope ({label})\nBlue = p < 0.05',
                            fontsize=11, fontweight='bold')
    axes[0, col].set_ylabel('Slope')
    axes[0, col].legend(fontsize=8)
    axes[1, col].bar(df_y['year'], df_y['r_squared'], color=colors, alpha=0.85)
    axes[1, col].axhline(y=r2_pool, color='tomato', linewidth=1.5, linestyle='--',
                         label=f'Pooled R2 ({r2_pool:.3f})')
    axes[1, col].set_title(f'Year-by-Year R2 ({label})', fontsize=11, fontweight='bold')
    axes[1, col].set_ylabel('R2')
    axes[1, col].set_xlabel('Year')
    axes[1, col].legend(fontsize=8)

plt.tight_layout()
plt.savefig('output_2c1_reg1_yearly.png', dpi=150)
plt.show()

print('v2 year-by-year:')
print(df_yearly_v2.to_string(index=False))
print('\nv5 year-by-year:')
print(df_yearly_v5.to_string(index=False))

### 2c.1.6 -- Specification Comparison Table: Regression 1

In [ ]:
sig_v2 = df_yearly_v2[df_yearly_v2['p_value'] < 0.05]
sig_v3 = df_yearly_v3[df_yearly_v3['p_value'] < 0.05]
sig_v4 = df_yearly_v4[df_yearly_v4['p_value'] < 0.05]
sig_v5 = df_yearly_v5[df_yearly_v5['p_value'] < 0.05]

print('\nRegression 1 -- Specification Comparison')
print(f'{"Specification":<45} {"Slope":>8} {"R2":>8} {"Sig years":>12}')
print('-' * 76)
rows = [
    ('v2 -- all years, geo regions only',       slope_v2, r2_v2, len(sig_v2), len(df_yearly_v2)),
    ('v3 -- non-geo + COVID excluded',          slope_v3, r2_v3, len(sig_v3), len(df_yearly_v3)),
    ('v4 -- non-geo + RJ excluded',             slope_v4, r2_v4, len(sig_v4), len(df_yearly_v4)),
    ('v5 -- non-geo + RJ + COVID excl. (star)', slope_v5, r2_v5, len(sig_v5), len(df_yearly_v5)),
]
for spec, slope, r2, sig, total in rows:
    print(f'{spec:<45} {slope:>8.4f} {r2:>8.4f} {str(sig)+"/"+str(total):>12}')
print('\n(star) = preferred structural estimate')

## 2c.2 -- Regression 2: Shipment Size vs Export Value Density

Testing whether states that ship in larger average quantities (kg per transaction)
tend to generate lower export value per kg -- the physical expression of the
commodity-industrial divide. A significant negative relationship would indicate
that bulk shipment size is systematically associated with lower value density
at the state level.

The regression is run on log scale for both variables given the range across states
(16,077 kg to 7,276,112 kg per transaction), and is examined in both a single-year
and pooled panel specification covering 1997-2025 to test stability over time.

Rio de Janeiro is flagged as a potential structural outlier -- crude oil generates
large shipments at moderate value density. Its influence is tested by comparing
results with and without the state.

In [ ]:
## Shipment size and value density -- all years, non-geographic excluded
query_shipsize = f"""
    SELECT e."CO_ANO" AS year,
           e."SG_UF_NCM" AS uf,
           u.nome_estado AS state,
           SUM(e."VL_FOB") AS exports_usd,
           SUM(e."KG_LIQUIDO") AS total_kg,
           COUNT(*) AS shipment_count
    FROM exp e
    JOIN uf u ON e."SG_UF_NCM" = u.sigla
    WHERE e."CO_ANO" <= {MAX_YEAR}
    AND e."KG_LIQUIDO" > 0
    AND u.nome_regiao NOT IN (
        'REGIAO NAO DECLARADA','CONSUMO DE BORDO',
        'MERCADORIA NACIONALIZADA','REEXPORTACAO'
    )
    GROUP BY e."CO_ANO", e."SG_UF_NCM", u.nome_estado
    ORDER BY year, exports_usd DESC
"""

df_ship = pd.read_sql(query_shipsize, engine)
df_ship['avg_kg_per_shipment'] = df_ship['total_kg']    / df_ship['shipment_count']
df_ship['avg_usd_per_kg']      = df_ship['exports_usd'] / df_ship['total_kg']
df_ship['exports_usd_bn']      = df_ship['exports_usd'] / 1e9
df_ship['log_avg_kg']          = np.log10(df_ship['avg_kg_per_shipment'].clip(lower=0.001))
df_ship['log_usd_per_kg']      = np.log10(df_ship['avg_usd_per_kg'].clip(lower=0.001))
df_ship = df_ship.merge(df_state[['uf', 'region']], on='uf', how='left')
df_ship['region'] = df_ship['region'].map(region_name_map).fillna(df_ship['region'])
df_ship = df_ship[df_ship['avg_kg_per_shipment'] > 0]

print(f'Shipment data loaded: {len(df_ship)} state-year observations')

### 2c.2.1 -- Pooled Regression: All States vs Excluding Rio de Janeiro

In [ ]:
df_full = df_ship.copy()
df_excl = df_ship[df_ship['uf'] != 'RJ'].copy()

slope_r2f, int_r2f, r_r2f, p_r2f, se_r2f = stats.linregress(df_full['log_avg_kg'], df_full['log_usd_per_kg'])
slope_r2e, int_r2e, r_r2e, p_r2e, se_r2e = stats.linregress(df_excl['log_avg_kg'], df_excl['log_usd_per_kg'])
r2_r2f = r_r2f ** 2
r2_r2e = r_r2e ** 2

x_r2 = np.linspace(df_full['log_avg_kg'].min(), df_full['log_avg_kg'].max(), 100)
y_r2f = slope_r2f * x_r2 + int_r2f
y_r2e = slope_r2e * x_r2 + int_r2e

fig, ax = plt.subplots(figsize=(13, 9))
df_lat_r2 = df_full[df_full['year'] == MAX_YEAR]
for region, group in df_full.groupby('region'):
    ax.scatter(group['log_avg_kg'], group['log_usd_per_kg'],
               color=REGION_COLORS.get(region, 'gray'), s=15, alpha=0.12, zorder=2)
for region, group in df_lat_r2.groupby('region'):
    ax.scatter(group['log_avg_kg'], group['log_usd_per_kg'],
               color=REGION_COLORS.get(region, 'gray'), s=60, alpha=0.95, zorder=4,
               edgecolors='black', linewidths=0.5, label=region)
    for _, row in group.iterrows():
        label = f"{row['uf']} (outlier)" if row['uf'] == 'RJ' else row['uf']
        ax.annotate(label, (row['log_avg_kg'], row['log_usd_per_kg']),
                    fontsize=7, ha='left', va='bottom')
ax.plot(x_r2, y_r2f, color='black', linewidth=1.8, linestyle='--',
        label=f'All states (R2={r2_r2f:.3f})', zorder=5)
ax.plot(x_r2, y_r2e, color='tomato', linewidth=1.8, linestyle='--',
        label=f'Excl. RJ (R2={r2_r2e:.3f})', zorder=5)
ax.set_title(
    f'Regression 2: Log(Avg Shipment Size) vs Log(USD/kg) (1997-{MAX_YEAR})\n'
    f'N={len(df_full)} observations  |  Slope={slope_r2f:.4f}  R2={r2_r2f:.3f}',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Log10 of Average kg per Shipment')
ax.set_ylabel('Log10 of Average USD per kg')
ax.legend(fontsize=8, title='Region')
plt.tight_layout()
plt.savefig('output_2c2_reg2_pooled.png', dpi=150)
plt.show()

print(f'Regression 2 Pooled Results (1997-{MAX_YEAR})')
print(f'  All states (n={len(df_full)}): Slope={slope_r2f:.4f}  R2={r2_r2f:.4f}  p={p_r2f:.6f}')
print(f'  Excl. RJ   (n={len(df_excl)}): Slope={slope_r2e:.4f}  R2={r2_r2e:.4f}  p={p_r2e:.6f}')

### 2c.2.2 -- Year-by-Year Regression: Regression 2

In [ ]:
yearly_r2 = []
for year, group in df_full.groupby('year'):
    if len(group) < 5: continue
    s_f, i_f, r_f, p_f, se_f = stats.linregress(group['log_avg_kg'], group['log_usd_per_kg'])
    group_e = group[group['uf'] != 'RJ']
    s_e, i_e, r_e, p_e, se_e = stats.linregress(group_e['log_avg_kg'], group_e['log_usd_per_kg'])
    yearly_r2.append({
        'year': year,
        'slope_full'   : round(s_f, 4), 'r2_full'    : round(r_f**2, 4), 'p_full'    : round(p_f, 4),
        'slope_excl_rj': round(s_e, 4), 'r2_excl_rj': round(r_e**2, 4), 'p_excl_rj': round(p_e, 4),
        'n': len(group)
    })
df_yearly_r2 = pd.DataFrame(yearly_r2)
sig_r2 = df_yearly_r2[df_yearly_r2['p_full'] < 0.05]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
colors_r2 = ['steelblue' if p < 0.05 else 'lightgray' for p in df_yearly_r2['p_full']]
ax1.bar(df_yearly_r2['year'] - 0.2, df_yearly_r2['slope_full'],
        width=0.4, color=colors_r2, alpha=0.85, label='All states')
ax1.bar(df_yearly_r2['year'] + 0.2, df_yearly_r2['slope_excl_rj'],
        width=0.4, color='tomato', alpha=0.5, label='Excl. RJ')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.axhline(y=slope_r2f, color='navy', linewidth=1.5, linestyle='--',
            label=f'Pooled slope ({slope_r2f:.4f})')
ax1.set_title('Year-by-Year Slope -- Log(Shipment Size) vs Log(USD/kg)\nBlue = p < 0.05',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Slope')
ax1.legend(fontsize=8)
ax2.bar(df_yearly_r2['year'] - 0.2, df_yearly_r2['r2_full'],
        width=0.4, color=colors_r2, alpha=0.85, label='All states')
ax2.bar(df_yearly_r2['year'] + 0.2, df_yearly_r2['r2_excl_rj'],
        width=0.4, color='tomato', alpha=0.5, label='Excl. RJ')
ax2.axhline(y=r2_r2f, color='navy', linewidth=1.5, linestyle='--',
            label=f'Pooled R2 ({r2_r2f:.3f})')
ax2.set_title('Year-by-Year R2 -- Regression 2', fontsize=12, fontweight='bold')
ax2.set_ylabel('R2')
ax2.set_xlabel('Year')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig('output_2c2_reg2_yearly.png', dpi=150)
plt.show()

print(f'Year-by-year Regression 2:')
print(df_yearly_r2.to_string(index=False))
print(f'\n  Significant years (p < 0.05): {len(sig_r2)} of {len(df_yearly_r2)}')
print(f'  Avg slope: {df_yearly_r2["slope_full"].mean():.4f}  Avg R2: {df_yearly_r2["r2_full"].mean():.4f}')

## 2c.3 -- Synthesis: Comparing Both Regressions

Comparing the two regressions across their preferred specifications to understand
what each contributes to the structural characterisation of Brazil's state-level
export competitiveness, and what the combination of both reveals that neither alone can show.

In [ ]:
print('\nRegression Comparison -- Preferred Specifications')
print(f'{"Metric":<40} {"Reg 1 (v5)":<22} {"Reg 2 (full)":<22}')
print('-' * 84)
rows = [
    ('Relationship tested',     'Log(Freq) vs HHI',      'Log(Size) vs Log(USD/kg)'),
    ('Preferred specification', 'Excl. RJ + COVID',      'All states'),
    ('N observations',          str(len(v5)),             str(len(df_full))),
    ('Pooled slope',            f'{slope_v5:.4f}',        f'{slope_r2f:.4f}'),
    ('Pooled R2',               f'{r2_v5:.4f}',           f'{r2_r2f:.4f}'),
    ('Significant years',       f'{len(sig_v5)}/25',      f'{len(sig_r2)}/29'),
    ('Primary outlier',         'Rio de Janeiro',         'Amazonas'),
    ('COVID impact',            'Breaks relationship',    'Weakens but holds'),
    ('Post-2015 trend',         'Weakening',              'Weakening'),
]
for label, v1_val, v2_val in rows:
    print(f'{label:<40} {v1_val:<22} {v2_val:<22}')

## 2c.4 -- Key Findings

### Finding 1 -- The Commodity-Industrial Divide is Structurally Encoded in Physical Trade Characteristics

Both regressions confirm that the commodity-industrial divide identified in Steps 2 and 2b
is not a cross-sectional snapshot -- it is a persistent structural feature that has held across
25-29 years of Brazilian state-level trade data. States that ship in large bulk quantities
generate systematically lower value per kg. States with high product concentration transact
less frequently. These relationships are stable, significant and mutually reinforcing.

### Finding 2 -- Shipment Size is a More Reliable Predictor Than Frequency

Regression 2 (R2 0.72, significant in 29 of 29 years) outperforms Regression 1
(R2 0.53 in preferred specification, significant in 25 of 25 years after outlier exclusion).
The physical dimension of trade -- what states ship and how valuable it is per kg -- is
a more stable structural signal than the operational dimension of how often they transact.

### Finding 3 -- Rio de Janeiro is a Structural Outlier in Regression 1 Only

Rio de Janeiro fits Regression 2 -- crude oil generates large shipments at moderate value
density exactly as the model predicts. But it breaks Regression 1 -- oil and gas operations
generate many transaction records despite extreme product concentration, violating the
frequency-HHI relationship. This distinction is analytically important: Rio de Janeiro's
anomaly is operational rather than structural.

### Finding 4 -- COVID-19 Affected Operations More Than Physical Trade Structure

COVID-19 broke Regression 1 (2020-2023 non-significant) but only weakened Regression 2
(significant throughout). Supply chain disruptions altered how states transacted but did not
fundamentally change what they shipped or how valuable it was per kg. This is consistent
with the Step 1 and Step 2 finding that Brazil's goods trade volumes were resilient
during COVID-19.

### Finding 5 -- The Post-2015 Weakening is the Most Unresolved Structural Finding

Both regressions show weakening explanatory power from approximately 2015-2016 onwards,
predating COVID and persisting through 2025. The mechanism cannot be confirmed from trade
flow data alone -- production data, port investment data and agricultural land use data
would be needed to investigate whether the MATOPIBA expansion, port modernisation or
commodity diversification in Norte/Nordeste states is the primary driver.

> Warning: The post-2015 weakening appearing simultaneously in both regressions predates
> COVID-19 and has not recovered to pre-2019 levels in either specification.
> This is the primary open question entering Steps 3-9 of the project.


### What Comes Next

**Municipality-Level Hotspots**
Disaggregating the state-level findings to the municipality level to identify the specific cities and industrial clusters driving each state's export profile.

**Product Complexity and Diversification**
NCM-level product analysis to explain the KSI and HHI findings — particularly Ceará's anomalous KSI (1.59) given the absence of an obvious single-commodity anchor, Pernambuco's high value density, and the RCA artefact problem in small states.

**Logistics Infrastructure**
Mapping the logistics profile classification against actual port infrastructure by state — testing whether current infrastructure matches or constrains each state's export development trajectory.